## Loading DS From Kaggle


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sunil123kumar/netflix-movies-and-tv-shows-dataset")

print("Path to dataset files:", path)

100%|██████████| 1.34M/1.34M [00:00<00:00, 72.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/sunil123kumar/netflix-movies-and-tv-shows-dataset/versions/1


In [8]:
import os

print(os.listdir(path))

['netflix_titles.csv']


## Load the DS in Spark


In [5]:
!pip install pyspark

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

Spark execute the entire plan only when we use the `show()` cmd until then it know just where is my ds and what i have to read


In [9]:
spark = SparkSession.builder \
    .appName("Netflix_ETL_Pipeline") \
    .getOrCreate()

csv_path = os.path.join(path, "netflix_titles.csv")

df = spark.read.csv(
    csv_path,
    header=True,    # Does not take headers as data
    inferSchema=True # Guesses the datatypes of the selected columns
)

df.show(5)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         NULL|Septem

Also unlike pandas spark does not have shape() as originally spark try's to run the ds parallely and when under a particular instance's execution it dosent really know is it running complete or partial ds hence there are the `.count() and .columns() ` mthd

In [10]:
print(f"Number of Rows : {df.count()}")
print(f"Number of Columns : {len(df.columns)}")

Number of Rows : 8809
Number of Columns : 12


Shows the schema of the ds

In [11]:
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



## Understanding the Schema & Data Quality Check

coz of `inferSchema=True` there may be case where spark infured a type incorrectly like how here in the `printSchema=true` it infered
> date as string

> release_date also as string

In [12]:
from pyspark.sql.functions import col, count, when
# Missing Vals
missing_values = df.select(
    [
      count(
          when(
              col(c).isNull(), c)
          ).alias(c)
      for c in df.columns
])

missing_values.show()

+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|      0|   1|    2|    2636| 826|    832|        13|           2|     6|       5|        3|          3|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



## Clean the Data


In [15]:
# Fill missing values in non-critical text columns
df = df.fillna({
    "director": "Unknown",
    "cast": "Unknown",
    "country": "Unknown"
})

# Remove rows missing essential information
df = df.dropna(subset=["title", "type", "release_year", "rating", "duration", "date_added", "listed_in", "description"])

## Verify

In [16]:
missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_values.show()

+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|      0|   0|    0|       0|   0|      0|         0|           0|     0|       0|        0|          0|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



Spark DataFrames are Immutable and hence we create new frames for each and then we reassign them

Duplicate records


In [17]:
print("Rows before removing duplicates:", df.count())

duplicate_count = df.count() - df.dropDuplicates().count()

print("Duplicate rows:", duplicate_count)

Rows before removing duplicates: 8788
Duplicate rows: 0


## MileStone


Dataset originally had 8807 rows

After dropping rows with missing essential values, it became `8788 rows`

Then

`Duplicate rows = 0`

This tells us something important:

> The Netflix dataset is clean with respect to duplicate records. **The only quality issues were missing values.**

## Transform

Convert Data Types

In [18]:
from pyspark.sql.functions import col

df = df.withColumn(
    "release_year",
    col("release_year").cast("integer")
)

Casted the data type of release_yr to int from string


In [19]:
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = false)
 |-- cast: string (nullable = false)
 |-- country: string (nullable = false)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



Rename the colm

In [21]:
df = df.withColumnRenamed(
    "listed_in",
    "genre"
)

print(df.columns)

['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genre', 'description']


In [22]:
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = false)
 |-- cast: string (nullable = false)
 |-- country: string (nullable = false)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- description: string (nullable = true)



## Feature Engineering : : Creating colms

created the colm of tv shows and movie

In [40]:
df = df.withColumns({
    "Movies": df.type == "Movie",
    "TV Shows": df.type == "TV Show"
})

create colm content age : current yr - content release yr

In [41]:
from pyspark.sql.functions import lit

current_yr = 2026

df = df.withColumn(
    "content_age",
    lit(current_yr) - col("release_year")
)

Verify the results for new colm

In [42]:
df.select(
    "title",
    "release_year",
    "content_age",
    "type",
    "Movies",
    "TV Shows"
).show(10, truncate=False)

+--------------------------------+------------+-----------+-------+------+--------+
|title                           |release_year|content_age|type   |Movies|TV Shows|
+--------------------------------+------------+-----------+-------+------+--------+
|Dick Johnson Is Dead            |2020        |6          |Movie  |true  |false   |
|Blood & Water                   |2021        |5          |TV Show|false |true    |
|Ganglands                       |2021        |5          |TV Show|false |true    |
|Jailbirds New Orleans           |2021        |5          |TV Show|false |true    |
|Kota Factory                    |2021        |5          |TV Show|false |true    |
|Midnight Mass                   |2021        |5          |TV Show|false |true    |
|My Little Pony: A New Generation|2021        |5          |Movie  |true  |false   |
|Sankofa                         |1993        |33         |Movie  |true  |false   |
|The Great British Baking Show   |2021        |5          |TV Show|false |tr

Cast the colms of Movies and TV Shows to int from bool

In [43]:
df = df.withColumns({
    "Movies": (col("type") == "Movie").cast("int"),
  "TV Shows": (col("type") == "TV Show").cast("int")
})

df.select("title", "Movies", "TV Shows").show(10, truncate=False)

+--------------------------------+------+--------+
|title                           |Movies|TV Shows|
+--------------------------------+------+--------+
|Dick Johnson Is Dead            |1     |0       |
|Blood & Water                   |0     |1       |
|Ganglands                       |0     |1       |
|Jailbirds New Orleans           |0     |1       |
|Kota Factory                    |0     |1       |
|Midnight Mass                   |0     |1       |
|My Little Pony: A New Generation|1     |0       |
|Sankofa                         |1     |0       |
|The Great British Baking Show   |0     |1       |
|The Starling                    |1     |0       |
+--------------------------------+------+--------+
only showing top 10 rows


Rename colms from movies to is_movie and TV Series to is_series so that we can know that these colms are bool

In [44]:
# Chaining both renaming
df = (
    df.withColumnRenamed("Movies", "is_movie")
      .withColumnRenamed("TV Shows", "is_series")
)

# Final OP

In [45]:
df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+--------+---------+-----------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|               genre|         description|is_movie|is_series|content_age|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+--------+---------+-----------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|             Unknown|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|       1|        0|          6|
|     s2|TV Show|       Blood & Water|             Unknown|Ama Qamata, Khosi...|        South Africa|Sep

## Conclusion

- In this experiment, an `ETL (Extract, Transform, Load) pipeline` was
successfully implemented using PySpark on the Netflix Movies and TV Shows dataset.
- The dataset was extracted from Kaggle, loaded into a Spark DataFrame, and explored to understand its structure and quality.
- **Data preprocessing** techniques such as handling missing values, checking for duplicate records, renaming columns, and converting data types were performed to improve data quality.
- Additionally, **new derived features** including *content_age, is_movie*, and *is_tv_show* were created to enhance the dataset for future analytical and machine learning tasks.
- Finally, the **cleaned and transformed** dataset was prepared for further analysis, demonstrating how PySpark can efficiently perform scalable data ingestion and ETL operations on real-world datasets.